# Quick Model Diagnostic

A lightweight version of the testing framework to quickly check feasibility and diagnostics.

In [13]:
import numpy as np
import pandas as pd
import time
import cvxpy as cp

from cma.data_reader import (
    read_vessel_class_data,
    read_port_data,
    read_sailing_distance_data,
    read_demand_with_transit_time,
    read_cnc_proforma_data,
)
from cma.port import PortGraph
from cma.servicegraph import ServiceGraph

np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## 1. Load Data (Minimal Subset)

In [14]:
vesselpool = read_vessel_class_data()
portpool_main, portpool_dmd = read_port_data()
dist_matrix = read_sailing_distance_data(portpool_main)
demand_matrix, transit_time_matrix = read_demand_with_transit_time(portpool_main)

portgraph = PortGraph(
    portpool_main, 
    dist_matrix, 
    demand_matrix,
    mat_transit_time=transit_time_matrix,
    filter_by_demand=False
)

proforma = read_cnc_proforma_data(portpool_main, vesselpool)
all_service_lines = proforma['lines']

# SUBSET FOR SPEED
service_lines = all_service_lines[:10]
servicegraph = ServiceGraph(service_lines)

trans_ports = portgraph.filtered_by_transship_capacity()
od_pairs_dict = servicegraph.get_all_paths(portgraph, trans_ports)

all_od_pairs = od_pairs_dict['od_pairs']
all_demands = od_pairs_dict['od_pairs_demand']
all_paths = od_pairs_dict['od_pairs_path']

filtered_pairs = []
filtered_paths = []
for od, paths, dmd in zip(all_od_pairs, all_paths, all_demands):
    if dmd > 0:
        filtered_pairs.append(od)
        filtered_paths.append(paths)

# SUBSET OD PAIRS
od_pairs = filtered_pairs[:50]
od_paths = filtered_paths[:50]

print(f"Loaded {len(service_lines)} lines and {len(od_pairs)} OD pairs for quick test.")

Loaded 10 lines and 50 OD pairs for quick test.


## 2. Run Optimization (All Features)

In [15]:
tuneparams = {
    'turnon-transship_shipclass_restriction': 1,
    'turnon-vessel_speed_optimization': 0,       # 0=ON (accurate)
    'turnon-port_operations_constraint': 1,
    'turnon-transit_time_penalty': 1,
    'ctrparam-kts_buffer': 0,
    'ctrparam-transship_A': 100,
    'ctrparam-speed_soft_cap_kts': 16.5,
    'ctrparam-speed_penalty_multiplier': 2.0,
    'ctrparam-transit_penalty_multiplier': 1000.0,
    'ctrparam-buffer_penalty_below_15pct': 1000.0,
    'ctrparam-buffer_penalty_above_30pct': 2000.0,
    'BigM-transship': 10000,
    'BigM-n_ships': 10,
    'BigM-saildays': 100,
    'BigM-line_capacity': 3000,
    'BigM-portcall_cost': 1e7,          
    'turnon-schedule_adherence': 1,     # Tether optimized arrival times to proforma
    'schedule_buffer_hrs': 120.0,        # 120 hour buffer for tethering
    'solver-MIPGap': 0.01,              
    'solver-TimeLimit': 21600,          
    'solver-MIPFocus': 1,               # Focus on finding feasible solutions quickly
    'solver-verbose': True              # Show Gurobi's progress logs
}

week_levels = [0.5, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [4]:
start_time = time.time()
solution = servicegraph.fulfill_demands(
    od_pairs,
    od_paths,
    portgraph,
    vesselpool,
    week_levels,
    tuneparams
)
end_time = time.time()

print(f"Solve Time: {end_time - start_time:.2f}s")
print(f"Total Cost: {solution['total cost']}")

c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 1 times so far.

  warnings.warn(msg, UserWarning)
c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 2 times so far.

  warnings.warn(msg, UserWarning)
c:\Use

                                     CVXPY                                     
                                     v1.7.5                                    


(CVXPY) Feb 25 12:54:41 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Feb 25 12:54:41 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Feb 25 12:54:41 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Feb 25 12:54:41 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Feb 25 12:54:42 PM: Compiling problem (target solver=GUROBI).
(CVXPY) Feb 25 12:54:42 PM: Reduction chain: CvxAttr2Constr -> Qp2SymbolicQp -> QpMatrixStuffing -> GUROBI
(CVXPY) Feb 25 12:54:42 PM: Applying reduction CvxAttr2Constr


-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Feb 25 12:54:42 PM: Applying reduction Qp2SymbolicQp
(CVXPY) Feb 25 12:54:43 PM: Applying reduction QpMatrixStuffing
(CVXPY) Feb 25 12:54:49 PM: Applying reduction GUROBI
(CVXPY) Feb 25 12:54:49 PM: Finished problem compilation (took 7.867e+00 seconds).
(CVXPY) Feb 25 12:54:49 PM: Invoking solver GUROBI  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
Set parameter Username
Set parameter LicenseID to value 2725074
Academic license - for non-commercial use only - expires 2026-10-20
Set parameter OutputFlag to value 1
Set parameter QCPDual to value 1
Set parameter MIPGap to value 0.01
Set parameter TimeLimit to value 21600
Set parameter MIPFocus to value 1
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 7 5700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  21600
MIPGap  0.01
MIPFocus  1
QCPDual  1

Optimize a model with 15347 rows, 7007 columns and 130539 nonzeros
Model fingerprint: 0x8b49e01b
Variable types: 4787 continuou

(CVXPY) Feb 25 12:55:29 PM: Problem status: optimal
(CVXPY) Feb 25 12:55:29 PM: Optimal value: 4.947e+09
(CVXPY) Feb 25 12:55:29 PM: Compilation took 7.867e+00 seconds
(CVXPY) Feb 25 12:55:29 PM: Solver (including time spent in interface) took 4.030e+01 seconds



Solve Time: 50.20s
Total Cost: 4946736686.412954


## 3. Diagnostics & Soft Constraint Analysis

In [5]:
if solution['total cost'] == float('inf'):
    print("❌ STILL INFEASIBLE. Checking hard constraints...")
    # ... (similar check as in test_buffer_constraint.ipynb) ...
else:
    print("✓ FEASIBLE! Analyzing soft constraint violations:")
    
    # Analyze Buffer Violations
    violation_lb = solution['buffer violation lb']
    violation_ub = solution['buffer violation ub']
    
    violation_data = []
    for i, line in enumerate(service_lines):
        lb_val = violation_lb[i].value if hasattr(violation_lb[i], 'value') else 0
        ub_val = violation_ub[i].value if hasattr(violation_ub[i], 'value') else 0
        if lb_val > 0.01 or ub_val > 0.01:
            violation_data.append({
                'Line': line.name(),
                'Below 15% (h)': lb_val,
                'Above 30% (h)': ub_val
            })
    
    if violation_data:
        print("\nBuffer Violations Detected:")
        print(pd.DataFrame(violation_data))
    else:
        print("\nNo buffer violations! All lines within 15-30% range.")

    # Analyze Weeks/Vessels picked
    weeks_vars = solution['weeks']
    picked_weeks = []
    for i, line in enumerate(service_lines):
        for j, wk in enumerate(week_levels):
            if weeks_vars[i, j].value > 0.5:
                picked_weeks.append({'Line': line.name(), 'Week': wk})
    
    print("\nCycle Times (Weeks) Picked:")
    print(pd.DataFrame(picked_weeks))

✓ FEASIBLE! Analyzing soft constraint violations:

Buffer Violations Detected:
      Line  Below 15% (h)  Above 30% (h)
0  BBX3CNC       0.000000      68.751154
1   BBXCNC       0.000000       2.321186
2   BMXCNC       0.000000      65.697957
3  CHN1CNC       0.000000      51.336779
4   CP3CNC       0.000000       2.359413
5   CP8CNC       0.000000      14.267312
6   CS1CNC      16.992492       0.000000

Cycle Times (Weeks) Picked:
      Line  Week
0  BBX2CNC     3
1  BBX3CNC     3
2   BBXCNC     2
3   BMXCNC     4
4  CHN1CNC     3
5  CMS2CNC     2
6   CP2CNC     2
7   CP3CNC     2
8   CP8CNC     1
9   CS1CNC     2


## 4. Full Dataset Evaluation

Testing the model on all service lines and all positive demand OD pairs to identify potential bottlenecks.

In [6]:
service_lines_full = all_service_lines
servicegraph_full = ServiceGraph(service_lines_full)

print("Finding all paths for full dataset...")
trans_ports = portgraph.filtered_by_transship_capacity()
od_pairs_dict_full = servicegraph_full.get_all_paths(portgraph, trans_ports)

all_od_pairs_full = od_pairs_dict_full['od_pairs']
all_demands_full = od_pairs_dict_full['od_pairs_demand']
all_paths_full = od_pairs_dict_full['od_pairs_path']

filtered_pairs_full = []
filtered_paths_full = []
for od, paths, dmd in zip(all_od_pairs_full, all_paths_full, all_demands_full):
    if dmd > 0:
        filtered_pairs_full.append(od)
        filtered_paths_full.append(paths)

print(f"Full Dataset: {len(service_lines_full)} lines and {len(filtered_pairs_full)} OD pairs.")

Finding all paths for full dataset...
Full Dataset: 31 lines and 741 OD pairs.


In [7]:
start_time = time.time()
solution_full = servicegraph_full.fulfill_demands(
    filtered_pairs_full,
    filtered_paths_full,
    portgraph,
    vesselpool,
    week_levels,
    tuneparams
)
end_time = time.time()

print(f"Full Solve Time: {end_time - start_time:.2f}s")
print(f"Total Cost: {solution_full['total cost']}")

c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 11 times so far.

  warnings.warn(msg, UserWarning)
c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 12 times so far.

  warnings.warn(msg, UserWarning)
c:\U

                                     CVXPY                                     
                                     v1.7.5                                    


(CVXPY) Feb 25 12:55:40 PM: Your problem has 32227 variables, 47991 constraints, and 0 parameters.
(CVXPY) Feb 25 12:55:41 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Feb 25 12:55:41 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Feb 25 12:55:41 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Feb 25 12:55:41 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Feb 25 12:55:44 PM: Compiling problem (target solver=GUROBI).
(CVXPY) Feb 25 12:55:44 PM: Reduction chain: CvxAttr2Constr -> Qp2SymbolicQp -> QpMatrixStuffing -> GUROBI
(CVXPY) Feb 25 12:55:44 PM: Applying reduction CvxAttr2Constr


-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Feb 25 12:55:47 PM: Applying reduction Qp2SymbolicQp
(CVXPY) Feb 25 12:55:53 PM: Applying reduction QpMatrixStuffing
(CVXPY) Feb 25 12:59:59 PM: Applying reduction GUROBI
(CVXPY) Feb 25 12:59:59 PM: Finished problem compilation (took 2.572e+02 seconds).
(CVXPY) Feb 25 12:59:59 PM: Invoking solver GUROBI  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
Set parameter OutputFlag to value 1
Set parameter QCPDual to value 1
Set parameter MIPGap to value 0.01
Set parameter TimeLimit to value 21600
Set parameter MIPFocus to value 1
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 7 5700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  21600
MIPGap  0.01
MIPFocus  1
QCPDual  1

Optimize a model with 47991 rows, 32227 columns and 584908 nonzeros
Model fingerprint: 0x65bd9c9a
Variable types: 25345 continuous, 6882 integer (6541 binary)
Coefficient statistics:
  Matrix range     [4e-05, 1e+07]
  Objective range  [1e-01, 1e+06]
  Bounds

(CVXPY) Feb 25 01:08:04 PM: Problem status: optimal
(CVXPY) Feb 25 01:08:04 PM: Optimal value: 2.595e+10
(CVXPY) Feb 25 01:08:04 PM: Compilation took 2.572e+02 seconds
(CVXPY) Feb 25 01:08:04 PM: Solver (including time spent in interface) took 4.850e+02 seconds


-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
Full Solve Time: 755.12s
Total Cost: 25945309416.58491


In [8]:
if solution_full['total cost'] == float('inf'):
    print("\n❌ FULL DATASET INFEASIBLE. Identifying problematic components...")
    
    # 1. Check for ports with zero productivity that have demand
    zero_prod_ports = []
    for port in portgraph.tolist_port():
        prods = port.get_producticity(vesselpool)
        if sum(prods) == 0:
            zero_prod_ports.append(port.get_id())
    
    if zero_prod_ports:
        print(f"\nPorts with ZERO productivity: {zero_prod_ports}")
        # Check if any OD pair involves these ports
        for i, (o_idx, d_idx) in enumerate(filtered_pairs_full):
            o_id = portgraph.get_port_by_idx(o_idx).get_id()
            d_id = portgraph.get_port_by_idx(d_idx).get_id()
            if o_id in zero_prod_ports or d_id in zero_prod_ports:
                print(f"  ⚠️ OD Pair {o_id}->{d_id} involves zero-productivity port.")

    # 2. Check for impossible distance/speed requirements (Hard Constraints)
    print("\nChecking for distance/speed violations:")
    for line in service_lines_full:
        dist = line.get_distance(portgraph)
        max_weeks = max(week_levels)
        min_speed_needed = dist / (24 * (7 * max_weeks - 1.0)) # 1 day min stay
        if min_speed_needed > 18.5:
            print(f"  ❌ Line {line.name()}: {dist:.0f}nm needs {min_speed_needed:.1f} kts at {max_weeks} weeks (Max 18.5)")

else:
    print("\n✓ FULL DATASET FEASIBLE!")
    
    # Violation Summary
    violation_lb = solution_full['buffer violation lb']
    violation_ub = solution_full['buffer violation ub']
    violation_summary = []
    for i, line in enumerate(service_lines_full):
        lb = violation_lb[i].value if hasattr(violation_lb[i], 'value') else 0
        ub = violation_ub[i].value if hasattr(violation_ub[i], 'value') else 0
        if lb > 0.1 or ub > 0.1:
            violation_summary.append({'Line': line.name(), 'LB_Violation': lb, 'UB_Violation': ub})
    
    if violation_summary:
        print("\nSignificant Buffer Violations in Full Dataset:")
        print(pd.DataFrame(violation_summary))


✓ FULL DATASET FEASIBLE!

Significant Buffer Violations in Full Dataset:
       Line  LB_Violation  UB_Violation
0   BBX3CNC      0.000000     68.751154
1    BBXCNC      0.000000      2.321186
2    BMXCNC      0.000000     64.258066
3    CP3CNC      0.000000      2.359413
4    CP8CNC      0.000000      4.042614
5    CS1CNC     26.036841      0.000000
6    CV8CNC      0.000000      4.472209
7    KCSCNC      8.600000      0.000000
8   SGS2CNC      0.000000      4.199653
9    SGSCNC      0.000000      0.199653
10   YSXCNC      0.000000      7.200149


## 5. Speed-Simplified Model (Full Dataset)

Running the model where vessel speed optimization is simplified (turnon-vessel_speed_optimization > 1/2).

In [9]:
tuneparams_simple = tuneparams.copy()
tuneparams_simple['turnon-vessel_speed_optimization'] = 1  # 1 = OFF (simplified)

start_time = time.time()
solution_simple = servicegraph_full.fulfill_demands(
    filtered_pairs_full,
    filtered_paths_full,
    portgraph,
    vesselpool,
    week_levels,
    tuneparams_simple
)
end_time = time.time()

print(f"Simple Model Solve Time: {end_time - start_time:.2f}s")
print(f"Total Cost: {solution_simple['total cost']}")

c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 42 times so far.

  warnings.warn(msg, UserWarning)
c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 43 times so far.

  warnings.warn(msg, UserWarning)
c:\U

                                     CVXPY                                     
                                     v1.7.5                                    


(CVXPY) Feb 25 01:08:14 PM: Your problem has 24991 variables, 21248 constraints, and 0 parameters.
(CVXPY) Feb 25 01:08:16 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Feb 25 01:08:16 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Feb 25 01:08:16 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Feb 25 01:08:16 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Feb 25 01:08:18 PM: Compiling problem (target solver=GUROBI).
(CVXPY) Feb 25 01:08:18 PM: Reduction chain: CvxAttr2Constr -> Qp2SymbolicQp -> QpMatrixStuffing -> GUROBI
(CVXPY) Feb 25 01:08:18 PM: Applying reduction CvxAttr2Constr


-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Feb 25 01:08:21 PM: Applying reduction Qp2SymbolicQp
(CVXPY) Feb 25 01:08:26 PM: Applying reduction QpMatrixStuffing
(CVXPY) Feb 25 01:11:54 PM: Applying reduction GUROBI
(CVXPY) Feb 25 01:11:54 PM: Finished problem compilation (took 2.174e+02 seconds).
(CVXPY) Feb 25 01:11:54 PM: Invoking solver GUROBI  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
Set parameter OutputFlag to value 1
Set parameter QCPDual to value 1
Set parameter MIPGap to value 0.01
Set parameter TimeLimit to value 21600
Set parameter MIPFocus to value 1
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 7 5700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  21600
MIPGap  0.01
MIPFocus  1
QCPDual  1

Optimize a model with 21248 rows, 24991 columns and 329238 nonzeros
Model fingerprint: 0x92378327
Variable types: 18667 continuous, 6324 integer (5983 binary)
Coefficient statistics:
  Matrix range     [4e-05, 1e+07]
  Objective range  [1e-01, 1e+06]
  Bounds

(CVXPY) Feb 25 01:12:01 PM: Problem status: optimal
(CVXPY) Feb 25 01:12:01 PM: Optimal value: 2.547e+10
(CVXPY) Feb 25 01:12:01 PM: Compilation took 2.174e+02 seconds
(CVXPY) Feb 25 01:12:01 PM: Solver (including time spent in interface) took 6.355e+00 seconds


-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
Simple Model Solve Time: 236.07s
Total Cost: 25474491501.456635


In [10]:
if solution_simple['total cost'] == float('inf'):
    print("\n❌ SIMPLE MODEL INFEASIBLE.")
else:
    print("\n✓ SIMPLE MODEL FEASIBLE!")
    
    # Violation Summary
    violation_lb = solution_simple['buffer violation lb']
    violation_ub = solution_simple['buffer violation ub']
    violation_summary = []
    for i, line in enumerate(service_lines_full):
        lb = violation_lb[i].value if hasattr(violation_lb[i], 'value') else 0
        ub = violation_ub[i].value if hasattr(violation_ub[i], 'value') else 0
        if lb > 0.1 or ub > 0.1:
            violation_summary.append({'Line': line.name(), 'LB_Violation': lb, 'UB_Violation': ub})
    
    if violation_summary:
        print("\nSignificant Buffer Violations in Simple Model:")
        print(pd.DataFrame(violation_summary))


✓ SIMPLE MODEL FEASIBLE!

Significant Buffer Violations in Simple Model:
     Line  LB_Violation  UB_Violation
0  CS1CNC     24.303844           0.0
1  KCSCNC      2.882909           0.0


## 6. Schedule Adherence & Transshipment Analysis

Analyze how closely optimized schedules follow the proforma tether, and examine transshipment wait times (enforcing the 1-day minimum).

In [11]:
print("--- Schedule Adherence (Tethering) Analysis ---")
if solution_simple['total cost'] != float('inf'):
    stay_days = solution_simple['port staying days'].value
    weeks_vars = solution_simple['weeks']
    
    adherence_data = []
    for i, line in enumerate(service_lines_full):
        proforma_sched = line.get_schedule(portgraph, vesselpool)
        if not proforma_sched: continue
        
        # Get picked week level
        wk_picked = 0
        for j, wk in enumerate(week_levels):
            if weeks_vars[i, j].value > 0.5: 
                wk_picked = wk
                break
        
        line_port_stay_days = np.sum(stay_days[i, :])
        line_sailing_days = 7 * wk_picked - line_port_stay_days
        
        anchor_wd, anchor_hr = line.get_anchor_eosp()
        base_eosp_days = (anchor_wd * 24 + anchor_hr) / 24.0
        total_dist = line.get_distance(portgraph)
        
        cum_dist = 0
        cum_stay_days = 0
        line_ports = line.tolist_port()
        
        for k, port in enumerate(line_ports):
            p_idx = portgraph.get_unique_index(port)
            num_visits = line_ports.count(port)
            
            if k > 0:
                cum_dist += portgraph.get_distance(line_ports[k-1], port)
            
            # Optimized ETB (days from Mon 00:00)
            opt_etb = base_eosp_days + (cum_dist / total_dist) * line_sailing_days + cum_stay_days
            prof_etb = proforma_sched[k][0] / 24.0
            
            # Deviation in hours
            deviation = (opt_etb - prof_etb) * 24.0
            
            adherence_data.append({
                'Line': line.name(),
                'Port': port.get_id(),
                'Opt ETB (h)': opt_etb * 24 % 168,
                'Prof ETB (h)': prof_etb * 24 % 168,
                'Dev (h)': deviation
            })
            
            cum_stay_days += stay_days[i, p_idx] / num_visits
            
    df_adherence = pd.DataFrame(adherence_data)
    print(f"\nAverage Schedule Deviation: {df_adherence['Dev (h)'].mean():.2f} hours")
    print(f"Max Schedule Deviation: {df_adherence['Dev (h)'].max():.2f} hours")
    
    print("\nLines with largest deviations:")
    print(df_adherence.sort_values('Dev (h)', ascending=False).head(10))

print("\n--- Transshipment Wait Time Analysis (Enforcing 1-Day Min) ---")
if solution_simple['total cost'] != float('inf'):
    ts_wait_data = []
    line_schedules = [line.get_schedule(portgraph, vesselpool) for line in service_lines_full]
    
    # Sample some transshipments from the paths
    count = 0
    for od_idx, paths in enumerate(filtered_paths_full):
        for path in paths:
            slots = path.tolist_slot()
            for i in range(len(slots) - 1):
                s1, s2 = slots[i], slots[i+1]
                if s1.get_service_name() != s2.get_service_name():
                    # Transshipment!
                    l1_idx = service_lines_full.index(s1.get_service())
                    l2_idx = service_lines_full.index(s2.get_service())
                    
                    seg1_idx = s1.get_service().get_segment_idx(s1.get_segment())
                    seg2_idx = s2.get_service().get_segment_idx(s2.get_segment())
                    
                    etd1 = line_schedules[l1_idx][seg1_idx][1]
                    etb2 = line_schedules[l2_idx][seg2_idx][0]
                    
                    wait_hrs = (etb2 - etd1) % 168.0
                    is_one_day_penalty = wait_hrs < 24.0
                    final_wait = wait_hrs + 168.0 if is_one_day_penalty else wait_hrs
                    
                    ts_wait_data.append({
                        'Hub': s1.get_end().get_id(),
                        'From': s1.get_service_name(),
                        'To': s2.get_service_name(),
                        'Raw Wait (h)': wait_hrs,
                        'Penalty Applied': is_one_day_penalty,
                        'Total Wait (h)': final_wait
                    })
                    count += 1
        if count > 50: break # Just a sample
        
    if ts_wait_data:
        print(pd.DataFrame(ts_wait_data).drop_duplicates().head(20))
    else:
        print("No transshipments found in sample paths.")

--- Schedule Adherence (Tethering) Analysis ---

Average Schedule Deviation: -103.66 hours
Max Schedule Deviation: 77.25 hours

Lines with largest deviations:
        Line   Port  Opt ETB (h)  Prof ETB (h)    Dev (h)
153   NPFCNC  JPHKT   162.645997          85.4  77.245997
154   NPFCNC  KRKAN    14.271639         109.4  72.871639
168  TIX2CNC  SGSIN   123.527480          65.2  58.327480
167  TIX2CNC  THLCH    54.431834         164.8  57.631834
169  TIX2CNC  IDJKT   163.179696         134.2  28.979696
98   JPXSCNC  JPNGO   115.474158          87.7  27.774158
116   JTXCNC  JPNGO    82.536296          75.5   7.036296
104  JTVSCNC  JPYOK   104.000000          99.2   4.800000
170  TIX2CNC  SGSIN    50.582766          46.0   4.582766
150   NPFCNC  JPSBS    22.348167          20.0   2.348167

--- Transshipment Wait Time Analysis (Enforcing 1-Day Min) ---
      Hub     From       To  Raw Wait (h)  Penalty Applied  Total Wait (h)
0   MYPKG   YCXCNC   CS2CNC          74.6            False      

C:\Users\ASUS\AppData\Local\Temp\ipykernel_23600\3203742648.py:37: RuntimeWarning: invalid value encountered in scalar divide
  opt_etb = base_eosp_days + (cum_dist / total_dist) * line_sailing_days + cum_stay_days


In [21]:
# Check unique ports in the full dataset using the available methods
if 'servicegraph_full' in locals():
    all_ports = set()
    # Iterate through all service lines in the graph
    for line in servicegraph_full.tolist_serviceLine():
        # Get the list of Port objects for this line
        for port in line.tolist_port():
            all_ports.add(port.get_id())
    
    print(f"Total unique ports in the FULL dataset (31 lines): {len(all_ports)}")
    # Optional: print the list of ports
    # print(sorted(list(all_ports)))

elif 'service_lines_full' in locals():
    # Fallback to the raw list of service lines if the graph wasn't built
    all_ports = set()
    for line in service_lines_full:
        for port in line.tolist_port():
            all_ports.add(port.get_id())
    print(f"Total unique ports in service_lines_full: {len(all_ports)}")

else:
    print("Variables 'servicegraph_full' or 'service_lines_full' not found. Please run the data loading cells.")

Total unique ports in the FULL dataset (31 lines): 56
